# Directional Derivatives on a Paraboloid

**Author:** Phabel Antonio Lopez-Delgado, BSc.

This notebook visualises directional derivatives on the paraboloid `z = x² + y²`. It highlights a point on the surface, then shows two unit direction vectors and the corresponding rates of change (directional derivatives) as line segments extending from the point. The interactive visualisation reinforces the geometric interpretation of the gradient as the direction of steepest ascent.

**Key Techniques & Libraries:**
- `numpy` – numerical computations and mesh generation
- `plotly.graph_objects` – interactive 3D visualisation
- Gradient computation and directional derivative evaluation
- Vector visualisation in 3D
- Interactive menus for layer toggling

**Data Source:**
Synthetic mathematical surface – no external data required.

**Note on Rendering:** If the interactive plots do not appear in the notebook, the figures are automatically saved as HTML files in the `plots/` directory. Open them in your web browser to view them.

In [1]:
# %% Imports and Configuration
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from pathlib import Path

# ---- Set Plotly Renderer ----
# For classic Jupyter Notebook: use 'notebook'
# For JupyterLab: use 'jupyterlab'
# For other environments: try 'iframe' or 'browser'
pio.renderers.default = 'notebook'  # Change as needed

# Reproducibility
np.random.seed(42)

# Ensure plots directory exists
Path("plots").mkdir(exist_ok=True)
print("Renderer set to:", pio.renderers.default)

Renderer set to: notebook


In [2]:
# %% Helper Functions
def create_paraboloid(x_range=(-10, 10), n_points=100):
    """
    Generate a grid for the paraboloid z = x^2 + y^2.

    Parameters:
    -----------
    x_range : tuple, (min, max) for x and y.
    n_points : int, number of points along each axis.

    Returns:
    --------
    tuple : (X, Y, Z) coordinate grids.
    """
    x = np.linspace(x_range[0], x_range[1], n_points)
    y = np.linspace(x_range[0], x_range[1], n_points)
    X, Y = np.meshgrid(x, y)
    Z = X**2 + Y**2
    return X, Y, Z

def gradient_paraboloid(x, y):
    """
    Gradient of f(x,y) = x^2 + y^2.

    Parameters:
    -----------
    x, y : float, coordinates.

    Returns:
    --------
    tuple : (df/dx, df/dy)
    """
    return (2*x, 2*y)

def directional_derivative(grad, unit_vec):
    """
    Compute directional derivative: D_u f = ∇f · u.

    Parameters:
    -----------
    grad : tuple, (df/dx, df/dy)
    unit_vec : np.ndarray, unit direction vector.

    Returns:
    --------
    float : directional derivative.
    """
    return grad[0]*unit_vec[0] + grad[1]*unit_vec[1]

def create_directional_derivative_trace(point, unit_vec, deriv_value, color='blue', name='Directional Derivative'):
    """
    Create a Scatter3d trace for the directional derivative segment.

    The segment extends from the point (x0, y0, z0) in the direction (ux, uy)
    with vertical change equal to the directional derivative.

    Parameters:
    -----------
    point : tuple, (x0, y0, z0) on the surface.
    unit_vec : np.ndarray, 2D unit direction vector.
    deriv_value : float, directional derivative.
    color : str, line colour.
    name : str, legend name.

    Returns:
    --------
    go.Scatter3d : trace object.
    """
    x0, y0, z0 = point
    x_end = x0 + unit_vec[0]
    y_end = y0 + unit_vec[1]
    z_end = z0 + deriv_value
    return go.Scatter3d(
        x=[x0, x_end],
        y=[y0, y_end],
        z=[z0, z_end],
        mode='lines',
        line=dict(color=color, width=4),
        name=name
    )

def create_interactive_figure(surface_data, point, traces_dd, title, buttons_config=None):
    """
    Create an interactive Plotly figure with toggle buttons.

    Parameters:
    -----------
    surface_data : tuple, (X, Y, Z) for the surface.
    point : tuple, (x0, y0, z0)
    traces_dd : list of go.Scatter3d traces for directional derivatives.
    title : str, figure title.
    buttons_config : list of dicts, optional custom buttons.

    Returns:
    --------
    plotly.graph_objects.Figure
    """
    # Surface trace
    surface_trace = go.Surface(
        x=surface_data[0], y=surface_data[1], z=surface_data[2],
        colorscale='Viridis', opacity=0.8,
        contours=dict(
            z=dict(show=True, usecolormap=True, highlightcolor="#42f462", project=dict(z=True))
        ),
        name='Paraboloid'
    )

    # Point trace
    point_trace = go.Scatter3d(
        x=[point[0]], y=[point[1]], z=[point[2]],
        mode='markers',
        marker=dict(size=8, color='red'),
        name='Point'
    )

    traces = [surface_trace, point_trace] + traces_dd
    n_dd = len(traces_dd)

    # Build default buttons if not provided
    if buttons_config is None:
        buttons = [
            dict(label='Show All', method='update',
                 args=[{'visible': [True]*len(traces)}, {'title': title}]),
            dict(label='Hide All', method='update',
                 args=[{'visible': [False]*len(traces)}, {'title': 'Nothing Visible'}]),
            dict(label='Surface Only', method='update',
                 args=[{'visible': [True, False] + [False]*n_dd}, {'title': 'Surface'}]),
            dict(label='Surface + Point', method='update',
                 args=[{'visible': [True, True] + [False]*n_dd}, {'title': 'Surface and Point'}]),
            dict(label='Surface + Derivatives', method='update',
                 args=[{'visible': [True, True] + [True]*n_dd}, {'title': 'With Derivatives'}])
        ]
        # Individual derivative toggles
        for i, trace in enumerate(traces_dd, start=2):
            vis = [True, True] + [False]*n_dd
            vis[i] = True
            buttons.append(dict(
                label=f'Surface + {trace.name}',
                method='update',
                args=[{'visible': vis}, {'title': f'Surface + {trace.name}'}]
            ))
    else:
        buttons = buttons_config

    fig = go.Figure(data=traces)
    fig.update_layout(
        title=dict(text=title, x=0.5, y=0.95, xanchor='center', yanchor='top',
                   font=dict(size=24, color='black')),
        scene=dict(xaxis_title='X', yaxis_title='Y', zaxis_title='Z'),
        legend=dict(orientation='h', yanchor='bottom', y=-0.2, xanchor='center', x=0.5),
        updatemenus=[
            dict(
                type='buttons',
                showactive=True,
                direction='down',
                x=0, y=1,
                xanchor='left', yanchor='top',
                buttons=buttons
            )
        ]
    )
    return fig

In [11]:
# %% Data Generation and Derivative Computation
# Surface
X, Y, Z = create_paraboloid()

# Point of interest
x0, y0 = 8, 8
z0 = x0**2 + y0**2
point = (x0, y0, z0)

# Gradient at the point
grad = gradient_paraboloid(x0, y0)

# Define two unit direction vectors
# u1 = [1, 1]/√2  (direction of steepest ascent-ish)
# u2 = [-1, 1]/√2 (perpendicular-ish direction)
u1 = np.array([1, 1]) / np.linalg.norm([1, 1])
u2 = np.array([-1, 1]) / np.linalg.norm([-1, 1])

# Compute directional derivatives
dd1 = directional_derivative(grad, u1)
dd2 = directional_derivative(grad, u2)

# Create traces for the directional derivatives
trace_dd1 = create_directional_derivative_trace(
    point, u1, dd1, color='blue', name='Directional Derivative u1 (along [1,1])'
)
trace_dd2 = create_directional_derivative_trace(
    point, u2, dd2, color='purple', name='Directional Derivative u2 (along [-1,1])'
)

# Print computed values for reference
print(f"Point: ({x0}, {y0}, {z0})")
print(f"Gradient: ({grad[0]:.2f}, {grad[1]:.2f})")
print(f"Directional derivative in u1: {dd1:.4f}")
print(f"Directional derivative in u2: {dd2:.4f}")
print(f"Gradient magnitude: {np.linalg.norm(grad):.4f}")

Point: (8, 8, 128)
Gradient: (16.00, 16.00)
Directional derivative in u1: 22.6274
Directional derivative in u2: 0.0000
Gradient magnitude: 22.6274


In [12]:
# %% Create Interactive Figure
fig = create_interactive_figure(
    surface_data=(X, Y, Z),
    point=point,
    traces_dd=[trace_dd1, trace_dd2],
    title="Directional Derivatives on a Paraboloid"
)

# Try to display; fallback to HTML
try:
    fig.show()
    print("Figure displayed in notebook.")
except Exception as e:
    print(f"Could not display interactively: {e}")
    fig.write_html("plots/directional_derivatives.html")
    print("Saved to plots/directional_derivatives.html – open in browser.")

Figure displayed in notebook.


## Summary and Next Steps

**Accomplished:**
- Visualised the paraboloid `z = x² + y²` with a selected point.
- Computed the gradient at that point and evaluated directional derivatives along two unit directions.
- Displayed the directional derivatives as line segments from the point, illustrating the rate of change of the surface in those directions.

**Key Insights:**
- The directional derivative is maximised in the direction of the gradient (steepest ascent).
- The directional derivative is zero in directions perpendicular to the gradient (level sets).
- The visualisation reinforces the concept of the gradient as a vector field pointing in the direction of greatest increase.

**Suggested Next Steps:**
1. **Gradient Vector Field:** Overlay a vector field of gradients across the entire surface.
2. **Interactive Point Selection:** Allow the user to click on the surface to compute derivatives interactively.
3. **Higher‑Order Derivatives:** Extend to second‑order directional derivatives and curvature analysis.
4. **Real‑World Application:** Apply to optimisation problems (e.g., gradient descent paths on loss surfaces).
5. **Symbolic Derivation:** Use `sympy` to derive the gradient symbolically for any input function.

This notebook bridges the gap between abstract calculus and tangible visualisation – a powerful tool for any data scientist or mathematician.